# Clean ancilla QROAM

QROAM loads a classical table using high address bits to select a block and low bits to route the selected cell to bank position zero. With multiple datasets or dimensions, the selected words are followed by the retained bank words. The target banks start clean.

The datasets share a selection shape. A plain QROM can also load shaped words with different target shapes, so its dataset shape is the selection shape followed by its own target shape. The clean-bank examples here load scalar words. The selection widths must hold every index and the target widths must hold every stored value. For unknown data, use the symbolic signature and resource declarations below.

Adapted from the Qualtran v0.7.0 QROAMClean tutorial. Copyright 2023 Google LLC. Licensed under Apache-2.0.

[Batched lookup and measurement-based erasure](https://arxiv.org/abs/1902.02134)

In [ ]:
from collections import Counter

from IPython.display import SVG, Markdown, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.clean_table_lookup import CleanTableLookup
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler.registers.lookup.qroam_clean_metadata import (
    permuted_qroam_erase_data,
    qroam_clean_signature,
)
from flagquantum.compiler.registers.lookup.qroam_resource_model import (
    source_qroam_clean_cost,
    source_qroam_measurement_erase_cost,
)
from flagquantum.compiler.registers.lookup.source_qroam_clean import QroamCleanTable
from flagquantum.compiler.registers.lookup.symbolic_qroam_resource_model import (
    source_blocked_lookup_resource_formula,
    source_shape_erasure_resource_formula,
)
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.dynamic import DynamicCircuit

## Concrete datasets and graphical signature

The two source examples load `0,1,…` and `1,2,…` over shapes `(5,)` and `(5,5)`. They use block lengths of two on each axis. Target words are dataset-specific, with the remaining cells exposed as junk.

In [ ]:
line = CleanTableLookup(QroamCleanTable((5,), (1,), (tuple(range(5)), tuple(range(1, 6)))), (3, 3))
grid = CleanTableLookup(QroamCleanTable((5, 5), (1, 1), (tuple(range(25)), tuple(range(1, 26)))), (5, 5))
display(SVG(field_signature_svg((
    ("QROAMClean: two datasets", line.source_signature, {}),
    ("QROAMClean: two axes", grid.source_signature, {}),
))))

In [ ]:
line_padded = permuted_qroam_erase_data(line.table)
grid_padded = permuted_qroam_erase_data(grid.table)
assert len(line_padded[0]) == 6 and len(grid_padded[0]) == 36
assert line_padded[0][-1] == (0, 4)
line_padded[0]

## Symbolic declarations and the call graph

For one selection axis, one dataset and zero controls, the source counts `ceil(N/K) + (K-1)b - 2` Toffoli equivalents for a forward block length `K` and `ceil(N/J) + J - 4` for an erasure block length `J`. The two block lengths can be chosen independently. Hardware instructions and explicit measurement slots belong to the separate executable path.

In [ ]:
n, m, a, b, k, log_columns, c = symbols("N M a b k l c", positive=True, integer=True)
one_axis = qroam_clean_signature((n,), (b,), (k,), controls=c)
two_axes = qroam_clean_signature((n, m), (a, b), (k, log_columns), controls=c)
assert len(tuple(one_axis)) == 4 and len(tuple(two_axes)) == 7
forward_model = source_blocked_lookup_resource_formula((n, m), (a, b), (k, log_columns), control_count=c)
erase_model = source_shape_erasure_resource_formula((n, m), (a, b), (k, log_columns), control_count=c)
forward_model, erase_model.wrapper

In [ ]:
cost = source_qroam_clean_cost(line.table, 0, line.selection_widths, line.target_widths)
graph = modeled_call_graph(cost, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(cost, max_depth=1))))
display(Markdown(format_call_graph_markdown(
    graph.root, graph.direct_calls, graph.leaf_counts, title=lambda child: child.name
)))

## Append and erase matching banks

Both operations use the same selected words and retained cells. The measured erasure first reverses the signed routing, then measures the bank bits and applies phase feedback. Its measurement count is reported separately from the declared adjoint cost.

In [ ]:
workspace_count = line.workspace_size()
selections = ((0, 1, 2),)
outputs = {"target0_": (3, 4, 5), "target1_": (6, 7, 8),
           "junk_target0_": (9, 10, 11), "junk_target1_": (12, 13, 14)}
workspace = tuple(range(15, 15 + workspace_count))
scratch_bits = tuple(range(workspace_count))
bank_bits = tuple(range(workspace_count, workspace_count + line.erase_result_count()))
args = dict(controls=(), selections=selections, outputs=outputs,
            workspace=workspace, classical_bits=scratch_bits)
circuit = DynamicCircuit(15 + workspace_count).append(line.make_on(**args))
forward = Counter(op.name for op in circuit.to_ir().instructions)
loaded = forward.total()
circuit.append(line.make_on(**args, bank_bits=bank_bits))
measured_erase = Counter(op.name for op in circuit.to_ir().instructions[loaded:])
assert measured_erase["measure"] >= len(bank_bits)
adjoint_cost = source_qroam_measurement_erase_cost(line.table, 0, line.target_widths)
{
    "emitted_forward": dict(forward),
    "emitted_erase": dict(measured_erase),
    "source_wrapper_toffoli": adjoint_cost.wrapper.gates.toffoli,
    "source_padded_leaf_toffoli": adjoint_cost.padded_leaf.gates.toffoli,
}